# Production scan: gamma_* = 1 to 20

Self-contained (matches notebook 17's pattern), built directly via
`ptbuilder.ic.generate_surrogate` rather than round-tripping through
notebook 15 -- notebook 15 stays a generic template with example values;
this notebook holds the actual production configuration and the reasoning
behind its node count / walltime, so both are reproducible later.

Readiness (see the long discussion this session): correctness confirmed via
identical-results reruns for GPU field evolution, GPU `phi2`, and the
`precompute_z_kernel` derivative-sharing split, all against the pre-port
binary, up to `gamma_ij~12.5`. Above that it's an algebraic rewrite of the
same validated math (no new numerics), so correctness is expected to carry
over, not separately re-checked here. Numerical convergence of the locked
production parameters at high `gamma_ij` was already established (see
`14_high_gamma_convergence_check.ipynb`'s context) -- not re-tested here
either. Format-B walltime-resume remains untested on real hardware; the
walltime margin below is sized partly to avoid ever needing it.

In [ ]:
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import generate_surrogate
from ptbuilder.physics import Phi4Potential, PhysicsModel

ROOT = Path(pt.__file__).parent.parent
LAMBDA_BAR = 0.84

model = PhysicsModel(Phi4Potential(LAMBDA_BAR))

GAMMA_STAR_MIN = 1.0
GAMMA_STAR_MAX = 20.0
KR_MIN, KR_MAX = 1.0, 100.0
N_OMEGA = 32   # production values, matches notebooks 15/17

## Sizing: node count and walltime

Row runtimes span 2-3 orders of magnitude across the grid (seconds at low
`gamma_ij` to hours near `gamma_ij_max=50`), so this isn't "how many nodes
can I afford" -- it's a bin-packing problem. Simulated LPT scheduling
(largest-`gamma_ij`-row first, assign each to the least-loaded GPU slot)
against a cost model calibrated on real measured runtimes from
`18_high_gamma_runtime_scaling.ipynb` (`gamma_ij=12,16,20,24`, fit within
~1-3%), using this scan's own domain-shared `T_MAX` (not the pair's-own-`R_*`
convention that notebook 18 itself uses -- see that notebook's discussion for
why the two give very different `n_t` for the same `gamma_ij`):

| nodes | slots | makespan | idle GPU-h | utilization |
|---|---|---|---|---|
| 2  | 8  | 23.7h | 0.1  | 100%  |
| 4  | 16 | 11.9h | 0.3  | 99.8% |
| **8**  | **32** | **6.0h**  | **2.7**  | **98.6%** |
| 10 | 40 | 5.9h  | 46.2 | 80.4% |
| 12 | 48 | 5.9h  | 93.4 | 67.0% |

Utilization stays ~99% through 8 nodes, then drops sharply -- past that
there aren't enough genuinely-long rows left to keep more slots busy, so
extra nodes buy idle allocation, not speed (makespan is pinned near the
single longest row's own ~5.9h regardless). **8 nodes (32 slots)** is the
sweet spot. Total estimated work: ~189.5 GPU-hours.

Largest-`gamma_ij`-first dispatch matters too, not just node count:
ascending order (the ORIGINAL default -- ordering fixed in both this
notebook and notebook 15 this session) gives 9.2h at 8 nodes vs. 6.0h
descending, for the identical work -- cheap rows exhaust early under
ascending order, leaving most of the pool idle waiting on the last few slow
ones.

Walltime set to ~2x the 6.0h estimate as margin against the untested
`gamma_ij>24` extrapolation (the most expensive rows, ~28% of total work,
sit past the calibrated range) and the untested Format-B resume.

In [ ]:
GPU_ACCOUNT  = 'm3166_g'
GPU_NODES    = 8            # see sizing rationale above
GPU_WALLTIME = '12:00:00'   # ~2x the 6.0h makespan estimate

## Build the scan

In [ ]:
scan = generate_surrogate(model, ROOT, f'lb{LAMBDA_BAR:g}', GAMMA_STAR_MIN, GAMMA_STAR_MAX,
                          KR_MIN, KR_MAX, N_OMEGA)
print(f'{len(scan.manifest_rows)} rows, gamma_ij {scan.gamma_ij_grid[0]:.2f}-'
      f'{scan.gamma_ij_grid[-1]:.2f}, T_MAX={scan.t_max:.1f}')

## SLURM script

In [ ]:
def write_slurm_script(scan, gpu_walltime, gpu_nodes, force_overwrite=False,
                       gpu_account='m3166_g', gpu_qos='premium'):
    """Writes scripts/runtime_scan_{tag}_gpu.sh for `scan` (from
    generate_surrogate) -- same pattern as notebooks 15/17, duplicated here
    since this is cluster-specific.

    Dispatches rows LARGEST-gamma_ij-first (descending), not ascending: see
    this notebook's sizing section for why -- ascending order leaves most of
    the pool idle at the end waiting on the last few slow rows. Row
    numbering/naming is untouched -- only the ORDER xargs -P pulls them in
    changes."""
    (ROOT / 'logs').mkdir(exist_ok=True)
    gpus_per_node = 4
    n_concurrent  = gpu_nodes * gpus_per_node
    n_tasks       = len(scan.manifest_rows)
    relative_manifest = scan.manifest.relative_to(ROOT)
    relative_output   = scan.output_root.relative_to(ROOT)
    overwrite_flag = ' --overwrite' if force_overwrite else ''

    script_path = ROOT / 'scripts' / f'runtime_scan_{scan.tag}_gpu.sh'
    script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{scan.tag}_gpu
#SBATCH --output={ROOT}/logs/bm_{scan.tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos={gpu_qos}
#SBATCH -A {gpu_account}
#SBATCH --nodes={gpu_nodes}
#SBATCH --ntasks={n_concurrent}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={gpu_walltime}
set -euo pipefail
REPO_ROOT="{ROOT}"  # baked in at generation time -- not $SLURM_SUBMIT_DIR/$PWD, which broke if sbatch was invoked from a different directory (e.g. scripts/)
cd "$REPO_ROOT"

run_one() {{
    local row_num=$1
    local line
    line=$(sed -n "$((row_num + 1))p" {relative_manifest})
    IFS=$'\t' read -r index gamma n_t setup name <<< "$line"
    output="{relative_output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {scan.n_min} --filon-panels-per-osc {scan.panels_per_oscillation} --s-batch-size {scan.s_batch_size}{overwrite_flag}
}}
export -f run_one

seq {n_tasks} -1 1 | xargs -P {n_concurrent} -I ROW bash -c 'run_one "$@"' _ ROW
''')
    script_path.chmod(0o755)
    print(f'Written: {script_path.relative_to(ROOT)}  '
          f'({n_tasks} tasks, {n_concurrent} concurrent slots on {gpu_nodes} nodes)'
          + (', --overwrite' if force_overwrite else ''))
    return script_path


script_path = write_slurm_script(scan, gpu_walltime=GPU_WALLTIME, gpu_nodes=GPU_NODES,
                                 gpu_account=GPU_ACCOUNT)  # gpu_qos defaults to 'premium'
print(f'Submit: sbatch {script_path.relative_to(ROOT)}')